# Spectral Scale Selection for Gaussian KDE

This notebook implements the experiment for samples
$X_1,\dots,X_n \sim N(0,\sigma^2)$ and Gaussian kernel density estimators.

For a bandwidth grid $0<h_1<\dots<h_N$, we build

$$
D_{2,ij}=\frac12\lVert \hat f_{h_i}-\hat f_{h_j}\rVert_2^2,\qquad
S_\gamma(\hat f_{h_i},\hat f_{h_j})=\exp(-\gamma D_{2,ij})
$$

using the exact Gaussian-kernel formula

$$
D_{2,ij}=\frac12\left(I_{ii}+I_{jj}-2I_{ij}\right),\qquad
I_{ij}=\frac{1}{n^2}\frac{1}{\sqrt{2\pi(h_i^2+h_j^2)}}
\sum_{k,l=1}^{n}\exp\left(-\frac{(X_k-X_l)^2}{2(h_i^2+h_j^2)}\right).
$$

The selected spectral bandwidth is the grid point where the Perron vector of $S_\gamma$ is maximal.
Since the true density is known in this simulation, we compare it to the oracle bandwidth

$$
h_{oracle}=\arg\min_{h_i}\int \left(\hat f_{h_i}(x)-f(x)\right)^2\,dx.
$$

The numerical implementation has been extracted into the nearby `kde_tools` package so that this notebook can focus on the mathematical workflow and experiments.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kde_tools import (
    exact_oracle_ise_values,
    gaussian_kde_values,
    gaussian_mixture_mean_and_variance,
    mixture_density,
    quadrature_beta2_divergence_matrix,
    quadrature_grid_for_kdes,
    quadrature_similarity_matrices,
    resolve_exponential_similarity_scale,
    sample_gaussian_mixture,
    spectral_choice_from_similarity,
)
from kde_tools.experiments import (
    run_one_all5_bandwidth_experiment,
    run_one_consistency_experiment as run_one_consistency_good_experiment,
    run_one_mixture_bandwidth_experiment as run_one_all5_mixture_bandwidth_experiment,
    run_one_spectral_vs_oracle_experiment,
)

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)


## Four similarity matrices used for spectral scale selection

For a bandwidth grid $0<h_1<\dots<h_N$, write
$\hat f_i=\hat f_{h_i}$.
The spectral selector needs a nonnegative similarity matrix, so large entries should mean that two KDEs are close as densities.

In this block the KDEs are evaluated on a one-dimensional quadrature grid $x_1,\dots,x_M$.
All integrals below are then approximated by the trapezoidal rule.
The quadrature grid is chosen wide enough that omitted tail mass is numerically negligible.

The four implemented similarities are

$$
S^{TV}_{ij}
=1-d_{TV}(\hat f_i,\hat f_j)
=\int \min\{\hat f_i(x),\hat f_j(x)\}\,dx,
$$

$$
S^{H}_{ij}
=\int \sqrt{\hat f_i(x)\hat f_j(x)}\,dx,
$$

$$
S^{\gamma}_{ij}
=\exp(-\gamma D_{2,ij}),\qquad
D_{2,ij}=\frac12\lVert \hat f_i-\hat f_j\rVert_2^2,
$$

and

$$
S^{JS}_{ij}
=1-\frac{\operatorname{JS}(\hat f_i,\hat f_j)}{\log 2}.
$$

Here

$$
\operatorname{JS}(f,g)
=H\left(\frac{f+g}{2}\right)-\frac12H(f)-\frac12H(g),
\qquad
H(f)=-\int f(x)\log f(x)\,dx.
$$


### Where the implementation lives

The notebook now contains the scientific narrative, experiment settings, summaries, and plots. The reusable numerical work is separated into small Python modules:

- `kde_tools/density.py`: KDE evaluation, quadrature grids, and exact Gaussian integrals;
- `kde_tools/bandwidths.py`: grid construction and classical bandwidth rules;
- `kde_tools/similarities.py`: similarities, Perron vectors, and spectral selection;
- `kde_tools/experiments.py`: one-repetition simulation functions used below.

This separation lets each experiment read from top to bottom without hiding long calculations inside a notebook cell. The functions remain ordinary Python and can be studied or tested independently.


## Exact Gaussian formulas

For the one-dimensional Gaussian kernel and the true density $N(0,\sigma^2)$, the oracle ISE
and the exponential beta-2 similarity matrix $S_\gamma$ can be computed exactly.
The TV-overlap, Hellinger-affinity, and JS-similarity matrices above are implemented by quadrature.

## Empirical distribution of chosen bandwidths

This follows the plotting style of the earlier KDE notebook: repeated independent samples give one
oracle index and one spectral index each, and the plot shows their relative frequencies on the same
bandwidth grid.

In [ ]:
# Pilot defaults. Increase num_simulations_distribution and n_h_distribution for final runs.
n_distribution = 200
sigma2_distribution = 1.0
num_simulations_distribution = 30
n_h_distribution = 60
base_seed_distribution = 90_000
gamma_distribution = "auto"

h_grid_distribution_center = np.sqrt(sigma2_distribution) * n_distribution ** (-1.0 / 5.0)
h_grid_distribution = np.geomspace(
    0.25 * h_grid_distribution_center,
    2.50 * h_grid_distribution_center,
    n_h_distribution,
)

bandwidth_distribution_results = [
    run_one_spectral_vs_oracle_experiment(
        seed=base_seed_distribution + simulation,
        n=n_distribution,
        sigma2=sigma2_distribution,
        h_grid=h_grid_distribution,
        gamma=gamma_distribution,
    )
    for simulation in range(num_simulations_distribution)
]

distribution_results = pd.DataFrame(bandwidth_distribution_results)

oracle_indices = distribution_results["oracle_index"].to_numpy(dtype=int)
spectral_indices = distribution_results["spectral_index"].to_numpy(dtype=int)

oracle_relative_frequencies = np.bincount(
    oracle_indices,
    minlength=len(h_grid_distribution),
) / num_simulations_distribution
spectral_relative_frequencies = np.bincount(
    spectral_indices,
    minlength=len(h_grid_distribution),
) / num_simulations_distribution

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    h_grid_distribution,
    oracle_relative_frequencies,
    marker="o",
    lw=2.3,
    label="oracle bandwidth",
)
ax.plot(
    h_grid_distribution,
    spectral_relative_frequencies,
    marker="o",
    lw=2.3,
    label=r"spectral $S_\gamma$ bandwidth",
)
ax.set_xscale("log")
ax.set_title(f"Empirical bandwidth distributions over {num_simulations_distribution} simulations")
ax.set_xlabel("bandwidth h")
ax.set_ylabel("relative frequency")
ax.legend()
plt.show()

exact_match_rate = np.mean(spectral_indices == oracle_indices)
within_1_rate = np.mean(np.abs(spectral_indices - oracle_indices) <= 1)
within_2_rate = np.mean(np.abs(spectral_indices - oracle_indices) <= 2)

print("Fixed-n oracle versus spectral experiment")
print("------------------------------------------")
print(f"Data distribution: X_i ~ N(0, {sigma2_distribution:g})")
print(f"Sample size per simulation: n = {n_distribution}")
print(f"Number of simulations: R = {num_simulations_distribution}")
print(f"Bandwidth grid size: n_h_distribution = {n_h_distribution}")
print(f"Gamma rule: {gamma_distribution}")
print()
print("Bandwidth summaries")
print("-------------------")
print(f"modal oracle h:   {h_grid_distribution[int(np.argmax(oracle_relative_frequencies))]:.6g}")
print(f"modal spectral h: {h_grid_distribution[int(np.argmax(spectral_relative_frequencies))]:.6g}")
print(f"median oracle h:  {distribution_results['oracle_h'].median():.6g}")
print(f"median spectral h:{distribution_results['spectral_h'].median():.6g}")
print(f"mean oracle h:    {distribution_results['oracle_h'].mean():.6g}")
print(f"mean spectral h:  {distribution_results['spectral_h'].mean():.6g}")
print()
print("Grid-index agreement")
print("--------------------")
print(f"Exact same grid index: {exact_match_rate:.2%}")
print(f"Within 1 grid step:   {within_1_rate:.2%}")
print(f"Within 2 grid steps:  {within_2_rate:.2%}")

distribution_results.head()


### Notes

- The only data-generating model in this notebook is $N(0,\sigma^2)$.
- The comparison target is the exact oracle bandwidth on the same finite grid, not Silverman's rule.
- `gamma="auto"` rescales each sample's distance matrix so the median positive distance maps to $e^{-1}$.
  Replace it by a fixed number to study sensitivity to the monotone transform.
- The repeated-bandwidth distribution plot intentionally mirrors the earlier KDE notebook: relative
  frequencies over bandwidth grid points, with oracle and spectral choices shown as two curves.

## KDEs selected by different similarity rules

This block draws the KDEs selected by the rules currently considered in the notebook:

- exact oracle ISE;
- JS quadrature;
- $S_\gamma(f,g)=\exp(-\gamma D_2(f,g))$;
- TV overlap $1-d_{TV}(f,g)=\int \min\{f,g\}$;
- Hellinger affinity $1-d_H(f,g)=\int \sqrt{fg}$.

The TV and Hellinger matrices are computed on the same quadrature grid as the JS matrix.

### One-sample comparison: selected bandwidths, Perron vectors, and selected KDEs

This block applies the five bandwidth choices on one simulated sample from $N(0,\sigma^2)$.

The oracle method is included as a benchmark. It is not a spectral method, because it uses the true density and minimizes

$$
\operatorname{ISE}(h_i)=\int\left(\hat f_{h_i}(x)-f(x)\right)^2\,dx.
$$

Therefore the oracle has no Perron vector. For the oracle panel below, the notebook plots the ISE curve instead.
For the four genuine spectral similarity matrices, the plotted vector is the Perron vector of the corresponding similarity matrix.

The four spectral choices are
$S^{TV}$, $S^H$, $S^\gamma$, and $S^{JS}$, as defined above.

In [ ]:
# One-sample comparison settings.
n_one_sample = 200
sigma2_one_sample = 1.0
seed_one_sample = 12345
n_h_one_sample = 80
m_x_one_sample = 2_001
gamma_one_sample = "auto"

rng = np.random.default_rng(seed_one_sample)
samples_one_sample = rng.normal(
    loc=0.0,
    scale=np.sqrt(sigma2_one_sample),
    size=n_one_sample,
)

h_grid_one_sample_center = np.sqrt(sigma2_one_sample) * n_one_sample ** (-1.0 / 5.0)
h_grid_one_sample = np.geomspace(
    0.25 * h_grid_one_sample_center,
    2.50 * h_grid_one_sample_center,
    n_h_one_sample,
)

x_grid_one_sample = quadrature_grid_for_kdes(
    samples=samples_one_sample,
    h_grid=h_grid_one_sample,
    m_x=m_x_one_sample,
)

density_values_one_sample = gaussian_kde_values(
    samples=samples_one_sample,
    h_grid=h_grid_one_sample,
    x_grid=x_grid_one_sample,
)
beta2_divergence_one_sample = quadrature_beta2_divergence_matrix(
    density_values=density_values_one_sample,
    x_grid=x_grid_one_sample,
)
resolved_gamma_one_sample = resolve_exponential_similarity_scale(
    beta2_divergence_one_sample,
    gamma_one_sample,
)
similarity_results_one_sample = {}
for similarity_name in (
    "tv_overlap",
    "hellinger_affinity",
    "exponential_l2_similarity",
    "js_similarity",
):
    current_similarity = quadrature_similarity_matrices(
        density_values=density_values_one_sample,
        x_grid=x_grid_one_sample,
        similarity_name=similarity_name,
        gamma=resolved_gamma_one_sample,
    )
    similarity_results_one_sample[similarity_name] = current_similarity
similarity_results_one_sample["gamma"] = resolved_gamma_one_sample

oracle_ise_one_sample = exact_oracle_ise_values(
    samples=samples_one_sample,
    h_grid=h_grid_one_sample,
    sigma2=sigma2_one_sample,
)
oracle_index_one_sample = int(np.argmin(oracle_ise_one_sample))

spectral_method_specs = [
    ("TV overlap", "tv_overlap"),
    ("Hellinger affinity", "hellinger_affinity"),
    (r"$S_\gamma$ from $D_2$", "exponential_l2_similarity"),
    ("JS similarity", "js_similarity"),
]

spectral_choices_one_sample = {}
for display_name, matrix_key in spectral_method_specs:
    choice = spectral_choice_from_similarity(similarity_results_one_sample[matrix_key])
    spectral_choices_one_sample[matrix_key] = {
        "display_name": display_name,
        "selected_index": int(choice["selected_index"]),
        "selected_h": float(h_grid_one_sample[int(choice["selected_index"])]),
        "perron_vector": choice["perron_vector"],
    }

true_density_one_sample = np.exp(-x_grid_one_sample**2 / (2 * sigma2_one_sample))
true_density_one_sample /= np.sqrt(2 * np.pi * sigma2_one_sample)

summary_rows_one_sample = [
    {
        "method": "Oracle ISE",
        "selected_index": oracle_index_one_sample,
        "selected_h": float(h_grid_one_sample[oracle_index_one_sample]),
        "uses_true_density": True,
        "has_perron_vector": False,
    }
]
for matrix_key in ["tv_overlap", "hellinger_affinity", "exponential_l2_similarity", "js_similarity"]:
    choice = spectral_choices_one_sample[matrix_key]
    summary_rows_one_sample.append(
        {
            "method": choice["display_name"],
            "selected_index": choice["selected_index"],
            "selected_h": choice["selected_h"],
            "uses_true_density": False,
            "has_perron_vector": True,
        }
    )

summary_one_sample = pd.DataFrame(summary_rows_one_sample)
print(f"Data: X_i ~ N(0, {sigma2_one_sample:g}), n = {n_one_sample}")
print(f"Bandwidth grid size: N_h = {n_h_one_sample}")
print(f"Quadrature grid size: M_x = {m_x_one_sample}")
print(f"Gamma used for S_gamma: {similarity_results_one_sample['gamma']:.6g}")
summary_one_sample


### Criterion curve and Perron vectors

The first panel shows the oracle ISE curve, where smaller is better.
The remaining four panels show the Perron vectors of the similarity matrices, where the selected bandwidth is the maximizer.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharex=True)

axes[0].plot(h_grid_one_sample, oracle_ise_one_sample, marker="o", lw=2)
axes[0].axvline(h_grid_one_sample[oracle_index_one_sample], color="crimson", ls="--", lw=2)
axes[0].set_title("Oracle ISE\nlower is better")
axes[0].set_ylabel(r"$\operatorname{ISE}(h)$")

for ax, (display_name, matrix_key) in zip(axes[1:], spectral_method_specs):
    choice = spectral_choices_one_sample[matrix_key]
    vector = choice["perron_vector"]
    selected_index = choice["selected_index"]
    ax.plot(h_grid_one_sample, vector, marker="o", lw=2)
    ax.axvline(h_grid_one_sample[selected_index], color="crimson", ls="--", lw=2)
    ax.set_title(f"{display_name}\nPerron vector")
    ax.set_ylabel("Perron weight")

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("bandwidth h")

fig.suptitle("Oracle criterion and spectral Perron-vector choices", y=1.05, fontsize=15)
plt.tight_layout()
plt.show()


### KDE at the selected bandwidth

Each panel shows the true density $N(0,\sigma^2)$ together with the KDE selected by the corresponding rule.

In [ ]:
selected_kde_specs_one_sample = [
    ("Oracle ISE", oracle_index_one_sample),
] + [
    (spectral_choices_one_sample[matrix_key]["display_name"], spectral_choices_one_sample[matrix_key]["selected_index"])
    for _, matrix_key in spectral_method_specs
]

fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)

for ax, (display_name, selected_index) in zip(axes, selected_kde_specs_one_sample):
    selected_h = h_grid_one_sample[selected_index]
    selected_kde = density_values_one_sample[selected_index]
    ax.plot(x_grid_one_sample, true_density_one_sample, color="black", lw=2.5, label="true density")
    ax.plot(x_grid_one_sample, selected_kde, lw=2.2, label="selected KDE")
    ax.set_title(f"{display_name}\nh = {selected_h:.4g}")
    ax.set_xlabel("x")
    ax.set_xlim(np.quantile(samples_one_sample, 0.005) - 1.5, np.quantile(samples_one_sample, 0.995) + 1.5)

axes[0].set_ylabel("density")
axes[0].legend(loc="upper right")
fig.suptitle("KDEs selected by the five bandwidth rules", y=1.05, fontsize=15)
plt.tight_layout()
plt.show()


## Empirical bandwidth distributions for all five rules

This repeats the bandwidth-selection experiment over 100 independent samples and plots the empirical
distribution of selected bandwidths for all five rules on the same bandwidth grid.

In [ ]:
# Empirical bandwidth distribution settings.
# Increase n_h_distribution_all5 or m_x_distribution_all5 for finer numerical resolution.
# Decrease num_simulations_distribution_all5 for a quick test run.
n_distribution_all5 = 200
sigma2_distribution_all5 = 1.0
num_simulations_distribution_all5 = 100
n_h_distribution_all5 = 80
m_x_distribution_all5 = 2_001
base_seed_distribution_all5 = 202_600
gamma_distribution_all5 = "auto"

h_grid_distribution_all5_center = np.sqrt(sigma2_distribution_all5) * n_distribution_all5 ** (-1.0 / 5.0)
h_grid_distribution_all5 = np.geomspace(
    0.25 * h_grid_distribution_all5_center,
    2.50 * h_grid_distribution_all5_center,
    n_h_distribution_all5,
)

all5_distribution_results = pd.DataFrame(
    [
        run_one_all5_bandwidth_experiment(
            seed=base_seed_distribution_all5 + simulation,
            n=n_distribution_all5,
            sigma2=sigma2_distribution_all5,
            h_grid=h_grid_distribution_all5,
            m_x=m_x_distribution_all5,
            gamma=gamma_distribution_all5,
        )
        for simulation in range(num_simulations_distribution_all5)
    ]
)

distribution_method_specs_all5 = [
    ("Oracle ISE", "oracle_index", "oracle_h"),
    ("TV overlap", "tv_index", "tv_h"),
    ("Hellinger affinity", "hellinger_index", "hellinger_h"),
    (r"$S_\gamma$ from $D_2$", "exponential_l2_similarity_index", "exponential_l2_similarity_h"),
    ("JS similarity", "js_index", "js_h"),
]

fig, ax = plt.subplots(figsize=(12.5, 6.0))

frequency_rows_all5 = []
summary_rows_all5 = []

for display_name, index_key, h_key in distribution_method_specs_all5:
    selected_indices = all5_distribution_results[index_key].to_numpy(dtype=int)
    relative_frequencies = np.bincount(
        selected_indices,
        minlength=len(h_grid_distribution_all5),
    ) / num_simulations_distribution_all5

    ax.plot(
        h_grid_distribution_all5,
        relative_frequencies,
        marker="o",
        lw=2.2,
        label=display_name,
    )

    for h_value, frequency in zip(h_grid_distribution_all5, relative_frequencies):
        if frequency > 0:
            frequency_rows_all5.append(
                {
                    "method": display_name,
                    "h": float(h_value),
                    "relative_frequency": float(frequency),
                }
            )

    modal_index = int(np.argmax(relative_frequencies))
    summary_rows_all5.append(
        {
            "method": display_name,
            "modal_h": float(h_grid_distribution_all5[modal_index]),
            "modal_relative_frequency": float(relative_frequencies[modal_index]),
            "median_h": float(all5_distribution_results[h_key].median()),
            "mean_h": float(all5_distribution_results[h_key].mean()),
        }
    )

ax.set_xscale("log")
ax.set_title(
    f"Empirical bandwidth distributions over {num_simulations_distribution_all5} simulations"
)
ax.set_xlabel("bandwidth h")
ax.set_ylabel("relative frequency")
ax.legend(ncol=2)
plt.show()

all5_frequency_table = pd.DataFrame(frequency_rows_all5)
all5_summary_table = pd.DataFrame(summary_rows_all5)

print("Empirical bandwidth distribution experiment")
print("-------------------------------------------")
print(f"Data distribution: X_i ~ N(0, {sigma2_distribution_all5:g})")
print(f"Sample size per simulation: n = {n_distribution_all5}")
print(f"Number of simulations: R = {num_simulations_distribution_all5}")
print(f"Bandwidth grid size: N_h = {n_h_distribution_all5}")
print(f"Quadrature grid size: M_x = {m_x_distribution_all5}")
print(f"Gamma rule: {gamma_distribution_all5}")
print()
all5_summary_table


## Consistency plot: sample size versus squared integrated error

This experiment studies whether the KDE selected by a bandwidth rule improves as the sample size grows.
For each sample size

$$
n=100,200,\dots,1000,
$$

we draw several independent samples from $N(0,\sigma^2)$.
For each sample we choose one bandwidth using

- the oracle ISE rule,
- the Hellinger-affinity spectral rule,
- the JS-similarity spectral rule.

The plotted quantity is the squared integrated error

$$
\lVert \hat f_{\hat h}-f\rVert_2^2
=\int \left(\hat f_{\hat h}(x)-f(x)\right)^2\,dx.
$$

Because the true density and the kernel are Gaussian in this notebook, this error is evaluated by the exact Gaussian formula.
The spectral choices themselves are still computed from the quadrature-based Hellinger and JS similarity matrices.

In [ ]:
# Consistency experiment settings.
# Runtime is roughly proportional to
# len(n_values_consistency_good) * num_simulations_consistency_good * n_h_consistency_good^2 * m_x_consistency_good.
n_values_consistency_good = np.arange(100, 1001, 100)
sigma2_consistency_good = 1.0
num_simulations_consistency_good = 20
n_h_consistency_good = 60
m_x_consistency_good = 1_501
base_seed_consistency_good = 303_000
gamma_consistency_good = "auto"

consistency_good_rows = []

for n_current in n_values_consistency_good:
    for simulation in range(num_simulations_consistency_good):
        seed = base_seed_consistency_good + 10_000 * int(n_current) + simulation
        consistency_good_rows.extend(
            run_one_consistency_good_experiment(
                seed=seed,
                n=int(n_current),
                sigma2=sigma2_consistency_good,
                h_grid=np.geomspace(
                    0.25 * np.sqrt(sigma2_consistency_good) * int(n_current) ** (-1.0 / 5.0),
                    2.50 * np.sqrt(sigma2_consistency_good) * int(n_current) ** (-1.0 / 5.0),
                    n_h_consistency_good,
                ),
                m_x=m_x_consistency_good,
                gamma=gamma_consistency_good,
            )
        )

consistency_good_results = pd.DataFrame(consistency_good_rows)

fig, ax = plt.subplots(figsize=(10, 5.5))

method_order_consistency_good = ["Oracle ISE", "Hellinger affinity", "JS similarity"]

for method in method_order_consistency_good:
    method_data = consistency_good_results[consistency_good_results["method"] == method]
    grouped = method_data.groupby("n")["squared_integrated_error"]
    n_plot = np.array(sorted(method_data["n"].unique()), dtype=int)
    median_error = grouped.median().loc[n_plot].to_numpy(dtype=float)
    q25_error = grouped.quantile(0.25).loc[n_plot].to_numpy(dtype=float)
    q75_error = grouped.quantile(0.75).loc[n_plot].to_numpy(dtype=float)

    ax.plot(n_plot, median_error, marker="o", lw=2.4, label=method)
    ax.fill_between(n_plot, q25_error, q75_error, alpha=0.18)

ax.set_title(
    "Sample size versus selected KDE squared integrated error\n"
    f"median and interquartile band over {num_simulations_consistency_good} simulations"
)
ax.set_xlabel("sample size n")
ax.set_ylabel(r"$\int (\hat f_{\hat h}(x)-f(x))^2\,dx$")
ax.set_yscale("log")
ax.legend()
plt.show()

consistency_good_summary = (
    consistency_good_results
    .groupby(["method", "n"])
    .agg(
        median_squared_integrated_error=("squared_integrated_error", "median"),
        q25_squared_integrated_error=("squared_integrated_error", lambda x: x.quantile(0.25)),
        q75_squared_integrated_error=("squared_integrated_error", lambda x: x.quantile(0.75)),
        median_selected_h=("selected_h", "median"),
    )
    .reset_index()
)

consistency_good_summary.head(12)


## Empirical bandwidth distributions for a two-component Gaussian mixture

This repeats the all-five bandwidth distribution experiment, but the data no longer come from a single Gaussian.
Instead, the sample is drawn from a deliberately non-uniform mixture

$$
X\sim w_1N(\mu_1,\sigma_1^2)+w_2N(\mu_2,\sigma_2^2),
$$

where the default parameters create one narrow peak and one much broader peak.
The oracle ISE is still computed exactly, because the kernel and the target density are both Gaussian mixtures.

In addition to the four spectral similarity selectors, this block also includes three classical KDE bandwidth rules:

- a direct AMISE plug-in rule, estimating $R(f'')=\int (f''(x))^2\,dx$ from a pilot KDE;
- least-squares cross-validation on the same bandwidth grid;
- Silverman's robust rule of thumb.

In [ ]:
# Two-Gaussian-mixture bandwidth distribution settings.
# The default mixture has one narrow peak and one broader, separated peak.
mixture_weights = np.array([0.60, 0.40], dtype=float)
mixture_means = np.array([-2.50, 3.00], dtype=float)
mixture_sigmas = np.array([0.35, 1.20], dtype=float)

n_distribution_mixture = 300
num_simulations_distribution_mixture = 100
n_h_distribution_mixture = 80
m_x_distribution_mixture = 2_001
base_seed_distribution_mixture = 404_000
gamma_distribution_mixture = "auto"

_, mixture_variance = gaussian_mixture_mean_and_variance(
    mixture_weights, mixture_means, mixture_sigmas
)
h_grid_distribution_mixture_center = (
    np.sqrt(mixture_variance) * n_distribution_mixture ** (-1.0 / 5.0)
)
h_grid_distribution_mixture = np.geomspace(
    0.08 * h_grid_distribution_mixture_center,
    2.50 * h_grid_distribution_mixture_center,
    n_h_distribution_mixture,
)

mixture_distribution_results = pd.DataFrame(
    [
        run_one_all5_mixture_bandwidth_experiment(
            seed=base_seed_distribution_mixture + simulation,
            n=n_distribution_mixture,
            h_grid=h_grid_distribution_mixture,
            weights=mixture_weights,
            means=mixture_means,
            sigmas=mixture_sigmas,
            m_x=m_x_distribution_mixture,
            gamma=gamma_distribution_mixture,
        )
        for simulation in range(num_simulations_distribution_mixture)
    ]
)

distribution_method_specs_mixture = [
    ("Oracle ISE", "oracle_index", "oracle_h"),
    ("TV overlap", "tv_index", "tv_h"),
    ("Hellinger affinity", "hellinger_index", "hellinger_h"),
    (r"$S_\gamma$ from $D_2$", "exponential_l2_similarity_index", "exponential_l2_similarity_h"),
    ("JS similarity", "js_index", "js_h"),
    ("Plug-in AMISE", "plugin_index", "plugin_h"),
    ("Least-squares CV", "lscv_index", "lscv_h"),
    ("Silverman rule", "silverman_index", "silverman_h"),
]

fig, ax = plt.subplots(figsize=(12.5, 6.0))

mixture_frequency_rows = []
mixture_summary_rows = []

for display_name, index_key, h_key in distribution_method_specs_mixture:
    selected_indices = mixture_distribution_results[index_key].to_numpy(dtype=int)
    relative_frequencies = np.bincount(
        selected_indices,
        minlength=len(h_grid_distribution_mixture),
    ) / num_simulations_distribution_mixture

    ax.plot(
        h_grid_distribution_mixture,
        relative_frequencies,
        marker="o",
        lw=2.2,
        label=display_name,
    )

    for h_value, frequency in zip(h_grid_distribution_mixture, relative_frequencies):
        if frequency > 0:
            mixture_frequency_rows.append(
                {
                    "method": display_name,
                    "h": float(h_value),
                    "relative_frequency": float(frequency),
                }
            )

    modal_index = int(np.argmax(relative_frequencies))
    mixture_summary_rows.append(
        {
            "method": display_name,
            "modal_h": float(h_grid_distribution_mixture[modal_index]),
            "modal_relative_frequency": float(relative_frequencies[modal_index]),
            "median_h": float(mixture_distribution_results[h_key].median()),
            "mean_h": float(mixture_distribution_results[h_key].mean()),
        }
    )

ax.set_xscale("log")
ax.set_title(
    f"Two-Gaussian-mixture bandwidth distributions over {num_simulations_distribution_mixture} simulations"
)
ax.set_xlabel("bandwidth h")
ax.set_ylabel("relative frequency")
ax.legend(ncol=2)
plt.show()

mixture_frequency_table = pd.DataFrame(mixture_frequency_rows)
mixture_summary_table = pd.DataFrame(mixture_summary_rows)

mixture_mean, mixture_variance = gaussian_mixture_mean_and_variance(
    mixture_weights,
    mixture_means,
    mixture_sigmas,
)

print("Two-Gaussian-mixture bandwidth distribution experiment")
print("------------------------------------------------------")
print("Data distribution:")
for k, (weight, mean, sigma) in enumerate(zip(mixture_weights, mixture_means, mixture_sigmas), start=1):
    print(f"  component {k}: weight={weight:g}, mean={mean:g}, sigma={sigma:g}")
print(f"Mixture mean: {mixture_mean:.6g}")
print(f"Mixture variance: {mixture_variance:.6g}")
print(f"Sample size per simulation: n = {n_distribution_mixture}")
print(f"Number of simulations: R = {num_simulations_distribution_mixture}")
print(f"Bandwidth grid size: N_h = {n_h_distribution_mixture}")
print(f"Quadrature grid size: M_x = {m_x_distribution_mixture}")
print(f"Gamma rule: {gamma_distribution_mixture}")
print()
mixture_summary_table


### KDEs at the most frequently selected mixture bandwidths

The previous block estimates the empirical distribution of selected bandwidths.
This block takes, for each method, the modal bandwidth from that empirical distribution and draws the corresponding KDE.

To make the comparison visually meaningful, all KDEs below are evaluated on the same fresh mixture sample.
The black curve is the true two-component mixture density.

In [ ]:
# Draw KDEs using the bandwidth selected most often by each method in the mixture experiment.
seed_modal_kde_mixture = 505_000
n_modal_kde_mixture = n_distribution_mixture

rng = np.random.default_rng(seed_modal_kde_mixture)
samples_modal_kde_mixture = sample_gaussian_mixture(
    rng=rng,
    n=n_modal_kde_mixture,
    weights=mixture_weights,
    means=mixture_means,
    sigmas=mixture_sigmas,
)

x_grid_modal_kde_mixture = quadrature_grid_for_kdes(
    samples=samples_modal_kde_mixture,
    h_grid=h_grid_distribution_mixture,
    m_x=m_x_distribution_mixture,
)
true_density_modal_kde_mixture = mixture_density(
    x_grid_modal_kde_mixture,
    weights=mixture_weights,
    means=mixture_means,
    sigmas=mixture_sigmas,
)

modal_bandwidth_rows_mixture = []
for display_name, index_key, h_key in distribution_method_specs_mixture:
    selected_indices = mixture_distribution_results[index_key].to_numpy(dtype=int)
    relative_frequencies = np.bincount(
        selected_indices,
        minlength=len(h_grid_distribution_mixture),
    ) / num_simulations_distribution_mixture
    modal_index = int(np.argmax(relative_frequencies))
    modal_bandwidth_rows_mixture.append(
        {
            "method": display_name,
            "modal_index": modal_index,
            "modal_h": float(h_grid_distribution_mixture[modal_index]),
            "modal_relative_frequency": float(relative_frequencies[modal_index]),
        }
    )

modal_bandwidth_table_mixture = pd.DataFrame(modal_bandwidth_rows_mixture)

n_methods_modal_mixture = len(modal_bandwidth_rows_mixture)
n_cols_modal_mixture = 4
n_rows_modal_mixture = int(np.ceil(n_methods_modal_mixture / n_cols_modal_mixture))
fig, axes = plt.subplots(
    n_rows_modal_mixture,
    n_cols_modal_mixture,
    figsize=(5.4 * n_cols_modal_mixture, 4.0 * n_rows_modal_mixture),
    sharey=True,
)
axes = np.asarray(axes).ravel()

for ax, row in zip(axes, modal_bandwidth_rows_mixture):
    method = row["method"]
    modal_h = row["modal_h"]
    modal_frequency = row["modal_relative_frequency"]
    kde_values = gaussian_kde_values(
        samples=samples_modal_kde_mixture,
        h_grid=np.array([modal_h]),
        x_grid=x_grid_modal_kde_mixture,
    )[0]

    ax.hist(
        samples_modal_kde_mixture,
        bins=35,
        density=True,
        alpha=0.18,
        color="gray",
        label="sample histogram",
    )
    ax.plot(x_grid_modal_kde_mixture, true_density_modal_kde_mixture, color="black", lw=2.5, label="true density")
    ax.plot(x_grid_modal_kde_mixture, kde_values, lw=2.2, label="KDE")
    ax.set_title(f"{method}\nh = {modal_h:.4g}, freq = {modal_frequency:.2f}")
    ax.set_xlabel("x")

for ax in axes[n_methods_modal_mixture:]:
    ax.axis("off")

axes[0].set_ylabel("density")
axes[0].legend(loc="upper right")
fig.suptitle("KDEs using the most frequently selected bandwidths in the mixture experiment", y=1.05, fontsize=15)
plt.tight_layout()
plt.show()

modal_bandwidth_table_mixture
